# PALABRIA – Notebook de ejecución

Este notebook permite ejecutar la aplicación PALABRIA de forma sencilla en Google Colab, y exponerlo mediante una URL pública.


Privacidad y datos:
- Cada usuario crea su propia base de datos.
- La base de datos NO se comparte con los autores del proyecto.


In [1]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Ejecuta esta celda SOLO si ya tienes la carpeta aplicacion_PALABRIA en tu Drive

import os, sys

PROJECT_PATH = "/content/drive/MyDrive/Colab Notebooks/EDUCACIÓN/EDUCACION"

if not os.path.exists(PROJECT_PATH):
    raise FileNotFoundError(
        "No se encontró la carpeta 'aplicacion_PALABRIA' en tu Drive.\n"
        "Copia ahí el proyecto o usa la celda de GitHub."
    )

os.chdir(PROJECT_PATH)
sys.path.append(PROJECT_PATH)

In [3]:
# Ejecuta esta celda SOLO si vas a descargar el proyecto desde GitHub en el sistema TEMPORAL de Colab

import os, sys

PROJECT_PATH = "/content/drive/MyDrive/Colab Notebooks/EDUCACIÓN/EDUCACION"
REPO_URL = "https://github.com/ncenteno-arch/APP-PALABRIA.git"


if not os.path.exists(PROJECT_PATH):
    !git clone $REPO_URL $PROJECT_PATH

os.chdir(PROJECT_PATH)
sys.path.append(PROJECT_PATH)

In [4]:
# Define la base de datos SQL
# La BD se guarda en tú Google Drive, independientemente de si el código viene de Drive o de GitHub

import pathlib

DB_PATH = "/content/drive/MyDrive/Colab Notebooks/EDUCACIÓN/EDUCACION/PALABRIA/data/palabria.db"
pathlib.Path(os.path.dirname(DB_PATH)).mkdir(parents=True, exist_ok=True)

os.environ["DB_PATH"] = DB_PATH

print("Base de datos personal configurada en:", DB_PATH)

Base de datos personal configurada en: /content/drive/MyDrive/Colab Notebooks/EDUCACIÓN/EDUCACION/PALABRIA/data/palabria.db


In [5]:
!pip install -r requirements.txt

  Using cached streamlit-1.54.0-py3-none-any.whl.metadata (9.8 kB)
Using cached streamlit-1.54.0-py3-none-any.whl (9.1 MB)


In [6]:
# Se descarga el modelo lingüístico de spaCy para español (necesario para la app)

!python -m spacy download es_core_news_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.0/568.0 MB 1.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [7]:
# Permite que el backend pueda quedarse funcionando en segundo plano, de forma que después podamos arrancar la interfaz web (frontend) sin que el notebook se bloquee

import nest_asyncio, logging

nest_asyncio.apply()
logging.basicConfig(level=logging.INFO)

In [30]:
# Se lanza el backend en local

import subprocess, time, os

os.chdir(PROJECT_PATH)

backend_process = subprocess.Popen(
    [
        "uvicorn",
        "backend.main:app",
        "--host", "127.0.0.1",
        "--port", "8000",
        "--reload",
        "--log-level", "warning"
    ]
)

time.sleep(15)
print("Backend activo")

Backend activo


In [31]:
# El frontend se conecta al backend local

import os

BACKEND_URL = "http://127.0.0.1:8000"

os.environ["BACKEND_URL"] = BACKEND_URL

print("Frontend conectado al backend")

Frontend conectado al backend


In [14]:
# Introduce TU token de ngrok (ngrok crea una dirección web pública temporal)

from pyngrok import ngrok

NGROK_TOKEN = input("Pega aquí tu NGROK AUTHTOKEN (no se guarda): ").strip()
ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()

Pega aquí tu NGROK AUTHTOKEN (no se guarda): 39G86ZiQCXiwOuX6xNMvFD8fx50_5CNfcpXaQqBg1dmHt6TRt


In [15]:
# Se lanza el frontend Streamlit y se muestra la URL visible externamente con ngrok

import subprocess

frontend_tunnel = ngrok.connect(8501)

subprocess.Popen([
    "streamlit", "run", "frontend/app.py",
    "--server.port=8501",
    "--server.headless=true",
    "--browser.serverAddress=0.0.0.0"
])

print("APLICACIÓN DISPONIBLE EN:")
print(frontend_tunnel.public_url)

APLICACIÓN DISPONIBLE EN:
https://phonophoric-smirchless-katheleen.ngrok-free.dev


**Recomendación:** lo más fiable es reiniciar el entorno de Colab para volver a ejecutar la aplicación.
Si no quieres reiniciar, prueba primero a limpiar los procesos en segundo plano.

In [ ]:
# Limpia los procesos anteriores (backend, frontend y ngrok)

import subprocess

subprocess.run(["pkill", "-f", "uvicorn"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(["pkill", "-f", "streamlit"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(["pkill", "-f", "ngrok"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)